In [17]:
from qrfc.quantum_random_forest import QuantumRandomForest, set_multiprocessing
from qrfc.split_function import SplitCriterion
from qrfc.data_construction import data_preprocessing
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics, datasets
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

In [18]:
data = pd.read_csv("dataset/by_antibiotic_12comorbdims/AMC.csv")
X = data.drop(columns=["anon_id","order_time_jittered_utc_shifted","resistant", 'Unnamed: 0'])
y = data["resistant"]
num_classes = 2

X = X.fillna(-9999999).astype(str)
encoder = OrdinalEncoder()
X = encoder.fit_transform(X)

In [19]:
train_set, test_set = data_preprocessing(X, y, X_dim=9)


New datapoint dimension: 9


In [20]:
n_qubits = 9                                         # Number of qubits for the embedding
dt_type = 'qke'                                      # Do not touch
ensemble_var = None                                  # Do not touch
branch_var = 'eff_anz_pqc_arch'                      # Type of Anzatz, or as a list for different down the tree - as given 
num_trees = 3                                        # Number of trees in ensemble 
split_num = 2                                        # Do not touch
pqc_sample_num = 2024                                # Number of circuit samples per kernel estimation
num_classes = num_classes                            # Number of classes in dataset
max_depth = 4                                        # Maximum depth of the tree
num_params_split = n_qubits*(n_qubits +1)            # Number of parameters in the embedding (different for different anzatz), list for different down the tree [2 * n_qubits ** 2 , n_qubits*(n_qubits +1), 2 * n_qubits ** 2]
num_rand_gen = 1                                     # Do not touch
num_rand_meas_q = n_qubits                           # Do not touch 
svm_num_train = 5                                    # L, Number of Landmarks
svm_c = 10                                           # C term in SVM optimisation, or list down the tree [100, 50, 20]
min_samples_split = svm_num_train                    # Minimum number of samples
embedding_type = 'as_params_all'                     # Type of embedding, or as a list - as given
criterion = SplitCriterion.init_info_gain('clas')    # Do not touch
device = 'cirq'                                      # Choose a device. Also possible to run on IBM

In [21]:
qrf = QuantumRandomForest(n_qubits, 'clas', num_trees, criterion, max_depth=max_depth, 
                          min_samples_split=min_samples_split, tree_split_num=split_num, num_rand_meas_q=num_rand_meas_q,
                          ensemble_var=ensemble_var, dt_type=dt_type, num_classes=num_classes, ensemble_vote_type='ave',
                          num_params_split=num_params_split, num_rand_gen=num_rand_gen, pqc_sample_num=pqc_sample_num,
                          embed=embedding_type, branch_var=branch_var, svm_num_train=svm_num_train, svm_c=svm_c, 
                          nystrom_approx=True, device=device)

In [ ]:
cores=5
set_multiprocessing(True, cores)
qrf.train(train_set, partition_sample_size=None)

In [ ]:
acc, pred_qrf = qrf.test(test_set, ret_pred=True, parallel=False, calc_tree_corr=True)

In [ ]:
print(f"Classification report for QRF:\n"
      f"{metrics.classification_report(test_set.y, pred_qrf)}\n")